### Import the API

Run the following cell to import the API into your session.

In [10]:
import ee
import pandas as pd
import time
import sys

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [2]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='cogent-coyote-500016-n3')

In [8]:
# ============================ CONFIG ======================================
PROJECT      = "cogent-coyote-500016-n3"          # <-- CHANGE THIS to your EE Cloud project
INPUT_XLSX   = r"cluster_date_climate.xlsx"   # input file (this script's folder)
OUTPUT_CSV   = "era5land_60day_temp.csv"     # output file
WINDOW_DAYS  = 60                             # exposure window length (days preceding interview)
BATCH_SIZE   = 200                            # points per Earth Engine request (200 is safe)
DATE_COL     = "interview_date"
LAT_COL      = "latnum"
LON_COL      = "longnum"
CLUST_COL    = "dhsclust"
# ERA5-Land daily aggregated; band temperature_2m is daily MEAN 2m temp in KELVIN
EE_DATASET   = "ECMWF/ERA5_LAND/DAILY_AGGR"
EE_BAND      = "temperature_2m"
# ==========================================================================

### Upload your local Excel file

Run the cell below to upload your `cluster_date_climate.xlsx` file. A file picker will appear, allowing you to select the file from your local machine. Once uploaded, the file will be accessible in the Colab environment.

In [5]:
from google.colab import files

uploaded = files.upload()

Saving cluster_date_climate.xlsx to cluster_date_climate.xlsx


After uploading, the file will be in the `/content/` directory (or the current working directory). You then need to update the `INPUT_XLSX` variable in the configuration cell to use the filename directly, as the file will no longer be at the Windows path.

In [6]:
# Verify the uploaded file is present
import os
print(os.listdir('.'))


['.config', 'cluster_date_climate.xlsx', 'sample_data']


In [11]:


def init_ee():
    """Initialise Earth Engine; authenticate on first run if needed."""
    try:
        ee.Initialize(project=PROJECT)
    except Exception:
        print("Authenticating to Earth Engine (browser will open)...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT)
    print("Earth Engine initialised on project:", PROJECT)


def build_feature(row):
    """Build an ee.Feature carrying the point geometry, the 60-day start/end, and an id."""
    end = ee.Date(str(pd.to_datetime(row[DATE_COL]).date()))          # interview date (exclusive upper bound handled below)
    start = end.advance(-WINDOW_DAYS, "day")                          # 60 days before
    geom = ee.Geometry.Point([float(row[LON_COL]), float(row[LAT_COL])])
    return ee.Feature(geom, {
        "row_id": int(row["row_id"]),
        "start": start.millis(),
        "end": end.millis(),
    })


def extract_batch(features):
    """
    For a batch of features, compute the 60-day mean temperature_2m at each point.
    Returns list of dicts: {row_id, era5land_t2m_60d (Celsius)}.
    """
    fc = ee.FeatureCollection(features)

    def per_feature(feat):
        feat = ee.Feature(feat)
        start = ee.Date(feat.get("start"))
        end = ee.Date(feat.get("end"))
        # ERA5-Land daily images over the 60-day window [start, end)
        col = (ee.ImageCollection(EE_DATASET)
               .select(EE_BAND)
               .filterDate(start, end))
        # mean over the time window, sampled at the point
        mean_img = col.mean()
        val = mean_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feat.geometry(),
            scale=11132,            # ~0.1 deg native ERA5-Land resolution in metres
            maxPixels=1e9,
        ).get(EE_BAND)
        return feat.set("t2m_kelvin", val)

    out = fc.map(per_feature)
    # pull results to the client
    data = out.getInfo()["features"]
    results = []
    for f in data:
        props = f["properties"]
        kelvin = props.get("t2m_kelvin", None)
        celsius = (kelvin - 273.15) if kelvin is not None else None
        results.append({
            "row_id": props["row_id"],
            "era5land_t2m_60d": round(celsius, 4) if celsius is not None else None,
        })
    return results


def main():
    init_ee()

    # ---- load input ----
    df = pd.read_excel(INPUT_XLSX)
    df = df.reset_index(drop=True)
    df["row_id"] = df.index
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    n = len(df)
    print(f"Loaded {n} cluster-date rows.")

    # ---- build features ----
    feats = [build_feature(r) for _, r in df.iterrows()]

    # ---- process in batches ----
    all_results = []
    n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE
    for b in range(n_batches):
        lo = b * BATCH_SIZE
        hi = min((b + 1) * BATCH_SIZE, n)
        batch_feats = feats[lo:hi]
        ok = False
        for attempt in range(1, 6):          # up to 5 retries per batch
            try:
                res = extract_batch(batch_feats)
                all_results.extend(res)
                ok = True
                print(f"  batch {b+1}/{n_batches}  rows {lo}-{hi-1}  OK")
                break
            except Exception as e:
                wait = attempt * 5
                print(f"  batch {b+1}/{n_batches} attempt {attempt} failed: {e}. "
                      f"retry in {wait}s...")
                time.sleep(wait)
        if not ok:
            print(f"  !! batch {b+1} FAILED after retries. Saving progress and stopping.")
            break
        time.sleep(1)   # gentle pause between batches

    # ---- merge & save ----
    res_df = pd.DataFrame(all_results)
    merged = df.merge(res_df, on="row_id", how="left")
    merged = merged.drop(columns=["row_id"])
    merged.to_csv(OUTPUT_CSV, index=False)

    got = merged["era5land_t2m_60d"].notna().sum()
    print(f"\nDone. Extracted {got}/{n} values.")
    if got < n:
        print(f"WARNING: {n-got} rows missing - rerun to fill gaps if needed.")
    print(f"Saved -> {OUTPUT_CSV}")
    print("\nSummary of 60-day mean temperature (Celsius):")
    print(merged["era5land_t2m_60d"].describe())


if __name__ == "__main__":
    main()

Earth Engine initialised on project: cogent-coyote-500016-n3
Loaded 1960 cluster-date rows.
  batch 1/10  rows 0-199  OK
  batch 2/10  rows 200-399  OK
  batch 3/10  rows 400-599  OK
  batch 4/10  rows 600-799  OK
  batch 5/10  rows 800-999  OK
  batch 6/10  rows 1000-1199  OK
  batch 7/10  rows 1200-1399  OK
  batch 8/10  rows 1400-1599  OK
  batch 9/10  rows 1600-1799  OK
  batch 10/10  rows 1800-1959  OK

Done. Extracted 1931/1960 values.
Saved -> era5land_60day_temp.csv

Summary of 60-day mean temperature (Celsius):
count    1931.000000
mean       27.087976
std         1.570843
min        22.422900
25%        26.053750
50%        27.786000
75%        28.261050
max        29.190600
Name: era5land_t2m_60d, dtype: float64


In [12]:
"""
============================================================================
Recover missing ERA5-Land values (coastal/border points) using a buffer
============================================================================
ERA5-Land has no data over water, so cluster points that fall just offshore
return null. This script re-extracts ONLY the missing rows using a circular
buffer around each point, so the reducer picks up the nearest land cell.

Run this AFTER extract_era5land.py, in the same folder.

INPUT : era5land_60day_temp.csv   (output of the main script, with 29 NaNs)
OUTPUT: era5land_60day_temp_filled.csv

Setup is identical to the main script (same PROJECT, same auth).
============================================================================
"""

import ee
import pandas as pd
import time

# ============================ CONFIG ======================================
PROJECT      = "cogent-coyote-500016-n3"     # your registered EE project
INPUT_CSV    = "era5land_60day_temp.csv"     # output of the main extraction
OUTPUT_CSV   = "era5land_60day_temp_filled.csv"
WINDOW_DAYS  = 60
DATE_COL     = "interview_date"
LAT_COL      = "latnum"
LON_COL      = "longnum"
VALUE_COL    = "era5land_t2m_60d"
BUFFER_M     = 15000                          # 15 km buffer to reach nearest land cell
EE_DATASET   = "ECMWF/ERA5_LAND/DAILY_AGGR"
EE_BAND      = "temperature_2m"
# ==========================================================================


def init_ee():
    try:
        ee.Initialize(project=PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT)
    print("Earth Engine initialised on project:", PROJECT)


def extract_point_buffer(lat, lon, interview_date):
    """60-day mean temperature_2m within a buffer around the point (mean over land cells)."""
    end = ee.Date(str(pd.to_datetime(interview_date).date()))
    start = end.advance(-WINDOW_DAYS, "day")
    pt = ee.Geometry.Point([float(lon), float(lat)])
    region = pt.buffer(BUFFER_M)
    col = (ee.ImageCollection(EE_DATASET)
           .select(EE_BAND)
           .filterDate(start, end))
    mean_img = col.mean()
    val = mean_img.reduceRegion(
        reducer=ee.Reducer.mean(),       # averages over whatever land cells fall in the buffer
        geometry=region,
        scale=11132,
        maxPixels=1e9,
    ).get(EE_BAND)
    info = val.getInfo()
    return (info - 273.15) if info is not None else None


def main():
    init_ee()
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    missing_mask = df[VALUE_COL].isna()
    miss = df[missing_mask]
    print(f"{len(miss)} missing rows to recover with a {BUFFER_M/1000:.0f} km buffer.")

    recovered = 0
    for idx, row in miss.iterrows():
        for attempt in range(1, 4):
            try:
                v = extract_point_buffer(row[LAT_COL], row[LON_COL], row[DATE_COL])
                if v is not None:
                    df.at[idx, VALUE_COL] = round(v, 4)
                    recovered += 1
                    print(f"  cluster {row['dhsclust']} {row[DATE_COL].date()}  -> {v:.3f} C")
                else:
                    print(f"  cluster {row['dhsclust']} {row[DATE_COL].date()}  still null (buffer too small?)")
                break
            except Exception as e:
                print(f"  retry {attempt} for cluster {row['dhsclust']}: {e}")
                time.sleep(attempt * 4)
        time.sleep(0.5)

    df.to_csv(OUTPUT_CSV, index=False)
    still_missing = df[VALUE_COL].isna().sum()
    print(f"\nRecovered {recovered} values. Still missing: {still_missing}.")
    print(f"Saved -> {OUTPUT_CSV}")
    print("\nFinal 60-day mean temperature (Celsius) summary:")
    print(df[VALUE_COL].describe())


if __name__ == "__main__":
    main()

Earth Engine initialised on project: cogent-coyote-500016-n3
29 missing rows to recover with a 15 km buffer.
  cluster 14 2022-10-17  -> 27.953 C
  cluster 14 2022-10-18  -> 27.907 C
  cluster 14 2022-10-19  -> 27.914 C
  cluster 20 2022-11-28  -> 25.598 C
  cluster 20 2022-11-29  -> 25.514 C
  cluster 20 2022-11-30  -> 25.436 C
  cluster 21 2022-11-16  -> 26.946 C
  cluster 21 2022-11-17  -> 26.879 C
  cluster 21 2022-11-18  -> 26.815 C
  cluster 37 2022-12-04  -> 25.045 C
  cluster 37 2022-12-05  -> 24.954 C
  cluster 37 2022-12-06  -> 24.862 C
  cluster 47 2022-10-14  -> 28.079 C
  cluster 47 2022-10-15  -> 28.081 C
  cluster 47 2022-10-16  -> 28.080 C
  cluster 48 2022-10-17  -> 28.039 C
  cluster 48 2022-10-18  -> 28.010 C
  cluster 48 2022-10-19  -> 28.013 C
  cluster 49 2022-11-22  -> 26.191 C
  cluster 49 2022-11-23  -> 26.090 C
  cluster 49 2022-11-24  -> 25.987 C
  cluster 51 2022-10-04  -> 27.989 C
  cluster 51 2022-10-05  -> 27.992 C
  cluster 51 2022-10-06  -> 27.992 C
  c